In [ ]:
import pandas as pd
import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi

def get_playlist_videos(playlist_url):
    """Lấy danh sách video_id và title từ link Playlist YouTube"""
    ydl_opts = {
        'extract_flat': True,  # Chỉ lấy metadata nhanh mà không tải video
        'skip_download': True,
        'quiet': True
    }

    videos = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(playlist_url, download=False)
        if 'entries' in info:
            for entry in info['entries']:
                if entry and entry.get('id'):
                    videos.append({
                        'id': entry['id'],
                        'title': entry.get('title', 'Unknown Title')
                    })
    return videos

def process_single_video(video_id, video_title, max_duration_hours=2):
    """Xử lý 1 video duy nhất: Lấy phụ đề, chia chunk 3 phút + overlap"""
    ytt_api = YouTubeTranscriptApi()

    try:
        transcript = ytt_api.fetch(video_id, languages=['en', 'vi'])
    except Exception:
        print(f"⏩ Bỏ qua (không có phụ đề/lỗi): {video_title} ({video_id})")
        return []

    raw_snippets = [{'start': s.start, 'duration': s.duration, 'text': s.text} for s in transcript]

    if not raw_snippets:
        return []

    # Kiểm tra độ dài < max_duration_hours
    total_duration_seconds = raw_snippets[-1]['start'] + raw_snippets[-1]['duration']
    if total_duration_seconds > max_duration_hours * 3600:
        print(f"⏩ Bỏ qua (dài {total_duration_seconds/3600:.1f}h > {max_duration_hours}h): {video_title}")
        return []

    # Chia chunks
    chunks = []
    chunk_duration = 180  # 3 phút
    overlap_words_count = 20
    current_chunk_words = []
    chunk_start_time = raw_snippets[0]['start']

    for i, item in enumerate(raw_snippets):
        words = item['text'].split()
        current_chunk_words.extend(words)

        if item['start'] - chunk_start_time >= chunk_duration or i == len(raw_snippets) - 1:
            chunk_text = " ".join(current_chunk_words)
            hrs = int(chunk_start_time // 3600)
            mins = int((chunk_start_time % 3600) // 60)
            secs = int(chunk_start_time % 60)

            chunks.append({
                "videoId": video_id,
                "title": video_title,
                "start": f"{hrs:02d}:{mins:02d}:{secs:02d}",
                "seconds": int(chunk_start_time),
                "text_chunk": chunk_text,
                "summary": ""
            })

            chunk_start_time = item['start']
            current_chunk_words = current_chunk_words[-overlap_words_count:] if len(current_chunk_words) > overlap_words_count else []

    print(f"✅ Đã xử lý xong ({len(chunks)} chunks): {video_title}")
    return chunks

def build_playlist_dataset(playlist_url, max_duration_hours=2):
    print("🔍 Đang lấy thông tin các video trong Playlist...")
    playlist_videos = get_playlist_videos(playlist_url)
    print(f"Tìm thấy tổng cộng {len(playlist_videos)} video trong danh sách phát.\n")

    all_chunks = []
    for index, video in enumerate(playlist_videos, 1):
        print(f"[{index}/{len(playlist_videos)}] Đang xử lý: {video['title']}")
        chunks = process_single_video(video['id'], video['title'], max_duration_hours)
        all_chunks.extend(chunks)

    df_result = pd.DataFrame(all_chunks)
    return df_result

# --- CHẠY THỬ VỚI PLAYLIST TRONG ẢNH CỦA BẠN ---
playlist_url = "https://www.youtube.com/playlist?list=PL-DKonjOZsHbZZ_xQC0HS46vBviytJJPg"

df_playlist = build_playlist_dataset(playlist_url, max_duration_hours=2)

# Lưu dataset thu được ra file JSON
df_playlist.to_json("playlist_dataset.json", orient="records", indent=4, force_ascii=False)
print(f"\n🎉 Hoàn thành! Tổng cộng tạo được {len(df_playlist)} chunks từ toàn bộ Playlist.")

🔍 Đang lấy thông tin các video trong Playlist...
Tìm thấy tổng cộng 58 video trong danh sách phát.

[1/58] Đang xử lý: All Machine Learning Algorithms in 23 Minutes
✅ Đã xử lý xong (9 chunks): All Machine Learning Algorithms in 23 Minutes
[2/58] Đang xử lý: Hướng dẫn sử dụng Claude Code từ A đến Z dành cho người mới
⏩ Bỏ qua (dài 2.5h > 2h): Hướng dẫn sử dụng Claude Code từ A đến Z dành cho người mới
[3/58] Đang xử lý: A Beginner's Guide to Using Claude Computer Use from A to Z
✅ Đã xử lý xong (34 chunks): A Beginner's Guide to Using Claude Computer Use from A to Z
[4/58] Đang xử lý: Comprehensive AI learning roadmap from A to Z in 2025
✅ Đã xử lý xong (7 chunks): Comprehensive AI learning roadmap from A to Z in 2025
[5/58] Đang xử lý: All about Neural Network in 12 minutes
✅ Đã xử lý xong (5 chunks): All about Neural Network in 12 minutes
[6/58] Đang xử lý: Summary of all Machine Learning algorithms (New version 2025)
✅ Đã xử lý xong (17 chunks): Summary of all Machine Learning algori

In [ ]:
data =  pd.read_json('/content/playlist_dataset.json')
print(data.head())


       videoId                                          title     start  \
0  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:00:00   
1  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:03:00   
2  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:06:02   
3  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:09:04   
4  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:12:06   

   seconds                                         text_chunk summary  
0        0  Xin chào các bạn Nếu các bạn là những người đa...          
1      180  vào phần Chính các bạn nhá linear regression l...          
2      362  ới với trình độ là CIO Giả sử chúng ta vẫn cố ...          
3      544  xem một con vật nó là con hổ hay là con sư tử ...          
4      726  ra sự kiện A Biết rằng sự kiện B đã xảy ra rồi...          


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 81 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (512 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started!")

Ollama server started!


In [ ]:
!ollama pull bge-m3

In [ ]:
!pip install -q ollama

In [ ]:
dataset = '/content/youtube_dataset.json'

df = pd.read_json(dataset)
print(df.head())

       videoId                                          title     start  \
0  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:00:00   
1  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:03:00   
2  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:06:02   
3  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:09:04   
4  ULE78ME1ckQ  All Machine Learning Algorithms in 23 Minutes  00:12:06   

   seconds                                         text_chunk  \
0        0  Xin chào các bạn Nếu các bạn là những người đa...   
1      180  vào phần Chính các bạn nhá linear regression l...   
2      362  ới với trình độ là CIO Giả sử chúng ta vẫn cố ...   
3      544  xem một con vật nó là con hổ hay là con sư tử ...   
4      726  ra sự kiện A Biết rằng sự kiện B đã xảy ra rồi...   

                                             summary  
0  Xin chào các bạn. Video này tổng hợp và tóm tắ...  
1  Linear regression là một thuật toán cơ bản tr

In [ ]:
import ollama

def generate_embeddings_in_batches(df, column_name='summary', batch_size=32):
    """
    Tối ưu sinh embedding theo batch thay vì dùng .apply() chạy từng dòng.
    """
    texts = df[column_name].tolist()
    all_embeddings = []

    # Chia danh sách text thành các batch nhỏ
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Gọi Ollama API cho cả batch
        response = ollama.embed(
            model='bge-m3',
            input=batch_texts
        )

        # Thêm kết quả vào danh sách tổng
        all_embeddings.extend(response.embeddings)
        print(f"Đã tạo embedding: {min(i + batch_size, len(texts))}/{len(texts)}")

    df['embedding_summary'] = all_embeddings
    return df

# Chạy tạo vector
df = generate_embeddings_in_batches(df, column_name='summary', batch_size=32)

# Kiểm tra chiều vector của bge-m3 (phải ra 1024)
print("Số chiều vector:", len(df['embedding_summary'].iloc[0]))

Đã tạo embedding: 32/510
Đã tạo embedding: 64/510
Đã tạo embedding: 96/510
Đã tạo embedding: 128/510
Đã tạo embedding: 160/510
Đã tạo embedding: 192/510
Đã tạo embedding: 224/510
Đã tạo embedding: 256/510
Đã tạo embedding: 288/510
Đã tạo embedding: 320/510
Đã tạo embedding: 352/510
Đã tạo embedding: 384/510
Đã tạo embedding: 416/510
Đã tạo embedding: 448/510
Đã tạo embedding: 480/510
Đã tạo embedding: 510/510
Số chiều vector: 1024


In [ ]:
df.to_json('/content/youtube_dataset_embedding.json', orient='records', indent=4, force_ascii=False)

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
import os
import pandas as pd
import numpy as np
import ollama
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

def load_dataset(source: str) -> pd.DataFrame:
    df = pd.read_json(source)
    # Bỏ text_chunk nếu không dùng đến, fillna để tránh lỗi chuỗi rỗng
    if 'text_chunk' in df.columns:
        df = df.drop(columns=['text_chunk'])
    return df.fillna('')

def calculate_cosine_similarity(vec1, vec2):
    """Hàm tính cosine similarity chuẩn hóa chiều dữ liệu NumPy"""
    a = np.array(vec1).reshape(1, -1)
    b = np.array(vec2).reshape(1, -1)
    # Lấy giá trị số thực float đầu tiên [[score]] -> score
    return float(sklearn_cosine_similarity(a, b)[0][0])

def get_video(dataset: pd.DataFrame, query: str, rows: int = 5, SIMILARITIES_RESULTS_THRESHOLD: float = 0.3) -> pd.DataFrame:
    dataset_copy = dataset.copy()


    response = ollama.embed(model='bge-m3', input=query)
    query_embedding = response.embeddings[0]

    # 2. Tính similarity score cho từng dòng
    dataset_copy['similarity_video'] = dataset_copy['embedding_summary'].apply(
        lambda x: calculate_cosine_similarity(query_embedding, x)
    )

    # 3. Lọc theo ngưỡng (Threshold)
    mask = dataset_copy['similarity_video'] >= SIMILARITIES_RESULTS_THRESHOLD
    dataset_copy = dataset_copy[mask].copy()

    # 4. Sắp xếp giảm dần theo điểm similarity
    dataset_copy = dataset_copy.sort_values(by='similarity_video', ascending=False)

    return dataset_copy.head(rows)

def display_results(videos: pd.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':\n")
    if videos.empty:
        print("Không tìm thấy video nào phù hợp với ngưỡng Similarity.")
        return

    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f"- {row['title']}")

        # Lấy 15 từ đầu tiên của summary để in tóm tắt ngắn
        summary_snippet = " ".join(str(row['summary']).split()[:15]) + "..."
        print(f"  Summary: {summary_snippet}")
        print(f"  YouTube: {youtube_url}")
        print(f"  Similarity: {row['similarity_video']:.4f}")  # Đã sửa lại tên cột chuẩn

        # Kiểm tra an toàn nếu có cột speaker thì mới in
        if 'speaker' in row and row['speaker']:
            print(f"  Speakers: {row['speaker']}")
        print("-" * 50)

In [ ]:
# 1. Load dataset đã tạo
df = load_dataset('/content/youtube_dataset_embedding.json')

# 2. Tìm kiếm thử
results = get_video(
    dataset=df,
    query="Linear regression là gì?",
    rows=3,
    SIMILARITIES_RESULTS_THRESHOLD=0.3
)

# 3. Hiển thị kết quả
display_results(results, query="Linear regression là gì?")


Videos similar to 'Linear regression là gì?':

- Summary of all Machine Learning algorithms (New version 2025)
  Summary: Linear Regression, hay hồi quy tuyến tính, là một trong những thuật toán cơ bản...
  YouTube: https://youtu.be/59XV4o8hOBk?t=0
  Similarity: 0.7347
--------------------------------------------------
- All Machine Learning Algorithms in 23 Minutes
  Summary: Linear regression là một thuật toán cơ bản trong machine learning dùng để dự đoán...
  YouTube: https://youtu.be/ULE78ME1ckQ?t=180
  Similarity: 0.7198
--------------------------------------------------
- A Comprehensive Guide to Popular ML Algorithms and DL Architectures in AI
  Summary: Logistic regression là một thuật toán phân loại nhị phân, trong khi multinomial logistic regression...
  YouTube: https://youtu.be/Hsalj5DMjnE?t=725
  Similarity: 0.5695
--------------------------------------------------


In [ ]:
print(df.columns)

Index(['videoId', 'title', 'start', 'seconds', 'summary'], dtype='object')


In [ ]:
pip install youtube_transcript_api

In [ ]:
pip install yt_dlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.7 MB/s  0:00:00
